# Missing values, duplicates and data types

**Video 6.3** | ShopKart project

**Goal:** Clean the raw ShopKart orders exactly like we did in Excel in Module 2 - but in a few lines of pandas - and save orders_clean.csv.

### How to open in Colab
1. Go to colab.research.google.com → **File → Upload notebook** → choose this `.ipynb` file.
2. Click the **folder icon** on the left sidebar (Files panel) and drag in the data files: `orders_raw.csv`, `products.csv` (used to look up the quantity-99 products).
3. Run cells one by one with **Shift + Enter**.

> Files you upload to the Files panel disappear when the runtime resets. If you see `FileNotFoundError`, upload them again.

## The cleaning plan
Every fix needs a **reason**. We never delete or fill data just to make the numbers look neat.

| Problem | Decision | Reason |
|---|---|---|
| 55 duplicate rows | Drop | Same order counted twice inflates revenue |
| City spellings | Strip, Title Case, rename old names | 'Bombay' and 'Mumbai' are the same market |
| unit_price "₹1,299" | Remove ₹ and comma, make number | Maths needs numbers |
| order_date two formats | Parse each exact format separately | Needed for months; the dayfirst shortcut silently flips ISO dates |
| payment_mode "upi" / blank | "upi"→"UPI", blank→"Unknown" | We do not know how they paid - do not guess |
| discount blank | Fill 0 | The business confirmed blank means no discount |
| returned Yes/No | Make Y/N | One code per answer |
| quantity 99 | Set to 1 | Data-entry typo; 10 Smartphones are real bulk orders, keep |
| rating blank | **Keep as missing** | "Not rated" is real information; filling would invent opinions |

In [1]:
import pandas as pd
df = pd.read_csv("orders_raw.csv")
print(df.shape)

(5055, 12)


## Step 1 - Duplicates

In [2]:
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()
print("After dropping:", df.shape)
print("Duplicate order_ids left:", df["order_id"].duplicated().sum())

Duplicate rows: 55
After dropping: (5000, 12)
Duplicate order_ids left: 0


## Step 2 - City names

In [3]:
print("Before:", df["city"].nunique(), "spellings")
df["city"] = df["city"].astype(str).str.strip().str.title()   # ' mumbai' -> 'Mumbai'
print("After strip + title:", df["city"].nunique())
df["city"] = df["city"].replace({"New Delhi": "Delhi", "Bangalore": "Bengaluru", "Bombay": "Mumbai",
                                 "Madras": "Chennai", "Calcutta": "Kolkata"})
print("After renaming:", df["city"].nunique())
df["city"].value_counts()

Before: 27 spellings
After strip + title: 15
After renaming: 10


city
Delhi        804
Mumbai       759
Bengaluru    736
Hyderabad    495
Pune         451
Chennai      442
Kolkata      414
Lucknow      326
Ahmedabad    303
Jaipur       270
Name: count, dtype: int64

## Step 3 - unit_price: text to number

In [4]:
print(df.loc[df["unit_price"].astype(str).str.contains("₹"), "unit_price"].head(3).tolist())
df["unit_price"] = (df["unit_price"].astype(str)
                    .str.replace("₹", "", regex=False)
                    .str.replace(",", "", regex=False)
                    .astype(float).astype(int))
print(df["unit_price"].dtype, df["unit_price"].min(), df["unit_price"].max())

['₹626', '₹2,551', '₹981']
int64 190 19948


## Step 4 - order_date: text to datetime
Two formats live in one column: most rows are `YYYY-MM-DD` (e.g. `2025-01-02`), 120 rows are `DD-MM-YYYY` (e.g. `02-01-2025`).

### The dayfirst trap
A tempting shortcut is `pd.to_datetime(df["order_date"], format="mixed", dayfirst=True)`. It runs without any error - but `dayfirst=True` is applied to **every** row, including the ISO ones. So `2025-01-02` (2 January) is read as **1 February**. No error, no warning, just wrong months.

In [5]:
# See the trap on three example dates
demo = pd.Series(["2025-01-02", "2025-03-04", "02-01-2025"])
print(pd.to_datetime(demo, format="mixed", dayfirst=True))   # 1st and 2nd are WRONG (Feb 1, Apr 3)

0   2025-02-01
1   2025-04-03
2   2025-01-02
dtype: datetime64[us]


In [6]:
# How many of OUR rows would the shortcut silently flip?
trap = pd.to_datetime(df["order_date"], format="mixed", dayfirst=True)
iso = df["order_date"].str.match(r"^\d{4}-")
flipped = (trap[iso].dt.strftime("%Y-%m-%d") != df.loc[iso, "order_date"]).sum()
print("ISO dates read with day and month swapped:", flipped, "of", len(df))

ISO dates read with day and month swapped: 1933 of 5000


Almost 2,000 orders would land in the wrong month - and every monthly chart and "best month" answer would be wrong.

### The safe way: one exact format at a time
1. Try the ISO format `%Y-%m-%d` on every row. Rows in the other format fail and become `NaT` (missing date) because of `errors="coerce"`.
2. Fill those gaps by trying `%d-%m-%Y`.

In [7]:
print("DD-MM-YYYY rows:", df["order_date"].str.contains(r"^\d{2}-").sum())
s_ = df["order_date"].astype(str)
df["order_date"] = pd.to_datetime(s_, format="%Y-%m-%d", errors="coerce").fillna(
                   pd.to_datetime(s_, format="%d-%m-%Y", errors="coerce"))
print(df["order_date"].dtype)
print("Dates that failed both formats:", df["order_date"].isnull().sum())
print(df["order_date"].min(), "to", df["order_date"].max())

DD-MM-YYYY rows: 120
datetime64[us]
Dates that failed both formats: 0
2025-01-01 00:00:00 to 2025-12-28 00:00:00


## Step 5 - payment_mode

In [8]:
df["payment_mode"] = df["payment_mode"].replace({"upi": "UPI"}).fillna("Unknown")
df["payment_mode"].value_counts()

payment_mode
UPI            2057
COD             983
Credit Card     888
Debit Card      685
Net Banking     297
Unknown          90
Name: count, dtype: int64

## Step 6 - discount: blank means 0

In [9]:
print("Blank discounts:", df["discount"].isnull().sum())
df["discount"] = df["discount"].astype(float).fillna(0.0)
print("After:", df["discount"].isnull().sum())

Blank discounts: 70
After: 0


## Step 7 - returned: one code

In [10]:
df["returned"] = df["returned"].replace({"Yes": "Y", "No": "N"})
df["returned"].value_counts()

returned
N    4313
Y     687
Name: count, dtype: int64

## Step 8 - quantity: typo or real outlier?

In [11]:
products = pd.read_csv("products.csv")
big = df[df["quantity"] >= 10].merge(products[["product_id", "product_name"]], on="product_id")
big[["order_id", "product_name", "quantity", "unit_price"]]

,order_id,product_name,quantity,unit_price
0,O10216,Smartphone,10,14999
1,O10791,Curtains Pair,99,1341
2,O11810,Water Bottle 1L,99,401
3,O13394,Perfume,99,1266
4,O13458,Smartphone,10,14999
5,O13825,Python Crash Course,99,893
6,O14392,Smartphone,10,14999


The 99s are cheap everyday items (a typo for 1). The three 10s are all Smartphones at ₹14,999 - a genuine bulk order. We fix the 99s and **keep** the 10s.

In [12]:
df.loc[df["quantity"] == 99, "quantity"] = 1
df["quantity"].value_counts().sort_index()

quantity
1     2307
2     1786
3      654
4      181
5       54
6       15
10       3
Name: count, dtype: int64

## Step 9 - rating: leave the gaps

In [13]:
print("Missing ratings:", df["rating"].isnull().sum())
print("Average rating (pandas skips blanks):", round(df["rating"].mean(), 2))

Missing ratings: 260
Average rating (pandas skips blanks): 3.96


If we filled blanks with 0, the average would fall and we would claim customers are unhappy when they simply did not rate. If we filled with the average, we would invent opinions. So we keep them as NaN.

## Final check and save

In [14]:
print(df.shape)
print(df.isnull().sum())
df.info()

(5000, 12)
order_id           0
order_date         0
customer_id        0
product_id         0
quantity           0
unit_price         0
discount           0
city               0
payment_mode       0
delivery_days      0
rating           260
returned           0
dtype: int64
<class 'pandas.DataFrame'>
Index: 5000 entries, 0 to 5054
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   order_id       5000 non-null   str           
 1   order_date     5000 non-null   datetime64[us]
 2   customer_id    5000 non-null   str           
 3   product_id     5000 non-null   str           
 4   quantity       5000 non-null   int64         
 5   unit_price     5000 non-null   int64         
 6   discount       5000 non-null   float64       
 7   city           5000 non-null   str           
 8   payment_mode   5000 non-null   str           
 9   delivery_days  5000 non-null   int64         
 10  rating         4

In [15]:
df.to_csv("orders_clean.csv", index=False)
print("Saved orders_clean.csv - download it from the Files panel (right-click -> Download)")

Saved orders_clean.csv - download it from the Files panel (right-click -> Download)


## Practice
Using the cleaned `df`: what share of orders now have payment_mode "Unknown"? How many orders came from Pune?

In [16]:
print(round((df["payment_mode"] == "Unknown").mean() * 100, 1), "% Unknown")
print((df["city"] == "Pune").sum(), "Pune orders")

1.8 % Unknown
451 Pune orders
